# M3 Module A: chuẩn bị 5 bảng đầu vào (Colab CPU)

Chạy lần lượt từng cell. Notebook chỉ **đọc** dữ liệu M1/M2 trên Drive; kết quả ghi vào thư mục mới trên `/content` rồi chép lên Drive `milestone 3/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
from pathlib import Path
# Thư mục chung arXiv-data (đường dẫn shortcut). Nếu bạn đã thêm lối tắt vào MyDrive thì dùng /content/drive/MyDrive/arXiv-data
DRIVE = Path('/content/drive/.shortcut-targets-by-id/1w9cVHms6Q9aakHKMSJY5DGzmBRAPleeT/arXiv-data')
assert (DRIVE / 'Full-data').is_dir() and (DRIVE / 'milestone 2').is_dir(), 'Sửa biến DRIVE cho đúng'
print(sorted(p.name for p in DRIVE.iterdir()))

In [ ]:
# Code: nhánh bạn đã push (đổi BRANCH nếu đặt tên khác)
BRANCH = 'huynh-nhi/m3-data-prep'
!rm -rf /content/repo && git clone --depth 1 -b {BRANCH} https://github.com/ChickenDev-cyber/arxiv-data-science /content/repo
!pip -q install pyarrow==23.0.1
%cd /content/repo
!git rev-parse HEAD
!PYTHONPATH=src python -m unittest tests.test_matching_data

In [ ]:
# Tự dò Full-data/*.zip và milestone 2/<owner>/<run>/handoff.json -> config nháp
!PYTHONPATH=src python -m arxiv_pipeline.matching.data scan --drive-root "{DRIVE}" --write /content/m3_sources.json

**Kiểm tra kỹ output trên**: mỗi owner phải có `m1_corpus` và `m2_run_dir`. Nếu một owner có nhiều run M2, hỏi nhóm run nào là chính thức rồi sửa `/content/m3_sources.json` (mở bằng thanh Files bên trái).

In [ ]:
# Chép 5 zip M1 về ổ local để đọc nhanh (khoảng 3,7 GB), rồi trỏ config sang bản local
import json, shutil
cfg_path = Path('/content/m3_sources.json')
cfg = json.loads(cfg_path.read_text())
local = Path('/content/m1'); local.mkdir(exist_ok=True)
for owner, spec in cfg['owners'].items():
    src = spec.get('m1_corpus')
    if src and src.startswith('/content/drive'):
        dst = local / Path(src).name
        if not dst.exists():
            print('copy', src); shutil.copy(src, dst)
        spec['m1_snapshot'] = Path(src).stem
        spec['m1_corpus'] = str(dst)
cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
print(json.dumps(cfg['owners'], ensure_ascii=False, indent=2))

In [ ]:
# Chạy chính. RUN_ID mới cho mỗi lần chạy; thư mục output phải chưa tồn tại.
RUN_ID = 'm3-inputs-01'
OUT = f'/content/{RUN_ID}'
!PYTHONPATH=src python -m arxiv_pipeline.matching.data prepare --config /content/m3_sources.json --output-dir {OUT}; echo exit=$?

Exit `0` = cả 5 owner `ok`; `2` = có owner thiếu/lỗi (xem dòng `[error]`, chi tiết trong `inventory.json`). Cảnh báo `[warning]` không chặn bàn giao nhưng cần ghi lại.

In [ ]:
# Sample nhỏ từ dữ liệu thật: 3 bài/owner, ưu tiên bài có cả bibtex và references
SAMPLE = f'/content/{RUN_ID}-sample'
!PYTHONPATH=src python -m arxiv_pipeline.matching.data sample --inventory {OUT}/inventory.json --output-dir {SAMPLE} --papers 3
!cd /content && zip -qr {RUN_ID}-sample.zip {RUN_ID}-sample && ls -lh /content/{RUN_ID}-sample.zip

In [ ]:
# Lưu lên Drive (thư mục mới, không ghi đè)
dest = DRIVE / 'milestone 3' / 'inputs' / RUN_ID
assert not dest.exists(), f'{dest} đã tồn tại, đổi RUN_ID'
shutil.copytree(OUT, dest)
shutil.copy(f'/content/{RUN_ID}-sample.zip', dest.parent / f'{RUN_ID}-sample.zip')
shutil.copy('/content/m3_sources.json', dest / 'm3_sources.json')
print('saved to', dest)